In [12]:
%pip install numpy scikit-fuzzy pytest

In [13]:
import os

os.makedirs("src", exist_ok=True)

with open("src/__init__.py", "w") as arquivo:
    pass

print("Estrutura do projeto criada!")

Estrutura do projeto criada!


In [26]:

%%writefile src/uti_triagem.py

import numpy as np
import skfuzzy as fuzzy
from skfuzzy import control as ctrl


def criar_sistema_triagem_uti():
    # 1. Criar as variaveis de entrada e de saida
    freq_cardiaca = ctrl.Antecedent(
        np.arange(40, 161, 1),
        "freq_cardiaca"
    )

    spo2 = ctrl.Antecedent(
        np.arange(70, 101, 1),
        "spo2"
    )

    risco = ctrl.Consequent(
        np.arange(0, 101, 1),
        "risco",
        defuzzify_method="centroid"
    )

    # 2. Funcoes de pertinencia da frequencia cardiaca
    freq_cardiaca["bradicardia"] = fuzzy.trimf(
        freq_cardiaca.universe, [40, 40, 60]
    )

    freq_cardiaca["normal"] = fuzzy.trimf(
        freq_cardiaca.universe, [50, 75, 100]
    )

    freq_cardiaca["taquicardia"] = fuzzy.trapmf(
        freq_cardiaca.universe, [90, 110, 160, 160]
    )

    # 3. Funcoes de pertinencia da saturacao
    spo2["critica"] = fuzzy.trapmf(
        spo2.universe, [70, 70, 88, 91]
    )

    spo2["limitrofe"] = fuzzy.trimf(
        spo2.universe, [89, 93, 96]
    )

    spo2["normal"] = fuzzy.trapmf(
        spo2.universe, [94, 97, 100, 100]
    )

    # 4. Funcoes de pertinencia do risco
    risco["baixo"] = fuzzy.trimf(
        risco.universe, [0, 0, 35]
    )

    risco["moderado"] = fuzzy.trimf(
        risco.universe, [25, 50, 75]
    )

    risco["critico"] = fuzzy.trapmf(
        risco.universe, [65, 85, 100, 100]
    )

    # 5. Criar as cinco regras clinicas
    regra1 = ctrl.Rule(
        spo2["critica"],
        risco["critico"]
    )

    regra2 = ctrl.Rule(
        freq_cardiaca["bradicardia"] & spo2["limitrofe"],
        risco["critico"]
    )

    regra3 = ctrl.Rule(
        freq_cardiaca["taquicardia"] & spo2["limitrofe"],
        risco["critico"]
    )

    regra4 = ctrl.Rule(
        freq_cardiaca["normal"] & spo2["limitrofe"],
        risco["moderado"]%0.8
    )

    regra5 = ctrl.Rule(
        freq_cardiaca["normal"] & spo2["normal"],
        risco["baixo"]
    )

    # 6. Compilar as regras no sistema
    sistema = ctrl.ControlSystem([
        regra1, regra2, regra3, regra4, regra5
    ])

    simulador = ctrl.ControlSystemSimulation(sistema)

    return simulador


def avaliar_paciente(simulador, bpm, spo2_val):
    # Validar a frequencia cardiaca
    if not 40 <= bpm <= 160:
        raise ValueError(
            "A frequencia cardiaca deve estar entre 40 e 160 bpm."
        )

    # Validar a saturacao
    if not 70 <= spo2_val <= 100:
        raise ValueError(
            "A saturacao deve estar entre 70% e 100%."
        )

    # Informar os valores ao simulador
    simulador.input["freq_cardiaca"] = bpm
    simulador.input["spo2"] = spo2_val

    # Executar a inferencia fuzzy
    simulador.compute()

    # Retornar o risco calculado
    return simulador.output["risco"]


if __name__ == "__main__":
    sim = criar_sistema_triagem_uti()
    print("Sistema de UTI instanciado com sucesso!")

    resultado = avaliar_paciente(sim, 75, 98)
    print(f"Risco calculado: {resultado:.2f}%")


Overwriting src/uti_triagem.py


In [27]:
from src.uti_triagem import (
    criar_sistema_triagem_uti,
    avaliar_paciente
)

simulador = criar_sistema_triagem_uti()

print("Sistema de UTI instanciado com sucesso!")

risco = avaliar_paciente(simulador, 75, 98)

print(f"Risco clínico calculado: {risco:.2f}%")

Sistema de UTI instanciado com sucesso!
Risco clínico calculado: 11.67%


In [28]:

%%writefile test_uti_triagem.py

import pytest

from src.uti_triagem import (
    criar_sistema_triagem_uti,
    avaliar_paciente
)


@pytest.fixture
def simulador_uti():
    return criar_sistema_triagem_uti()


def test_paciente_saudavel_baixo_risco(simulador_uti):
    # Paciente com FC normal e saturacao de 98%
    risco = avaliar_paciente(simulador_uti, 75, 98)

    assert risco < 30.0


def test_salto_nao_linear_queda_spo2(simulador_uti):
    # Comparar duas saturacoes
    risco_95 = avaliar_paciente(simulador_uti, 75, 95)
    risco_89 = avaliar_paciente(simulador_uti, 75, 89)

    assert risco_95 < 35.0
    assert risco_89 > 65.0
    assert risco_89 > risco_95


def test_validacao_limites_fisiologicos(simulador_uti):
    # FC abaixo do minimo permitido
    with pytest.raises(ValueError):
        avaliar_paciente(simulador_uti, 39, 98)

    # SpO2 acima do maximo permitido
    with pytest.raises(ValueError):
        avaliar_paciente(simulador_uti, 75, 101)


Overwriting test_uti_triagem.py


In [29]:
!python -m pytest -v test_uti_triagem.py

============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: platformdirs-4.13.5, anyio-4.15.1, typeguard-4.6.0, langsmith-0.14.12
collected 3 items                                                              

test_uti_triagem.py::test_paciente_saudavel_baixo_risco PASSED           [ 33%]
test_uti_triagem.py::test_salto_nao_linear_queda_spo2 PASSED             [ 66%]
test_uti_triagem.py::test_validacao_limites_fisiologicos PASSED          [100%]

============================== 3 passed in 1.57s ===============================


In [21]:
sim = criar_sistema_triagem_uti()

risco_95 = avaliar_paciente(sim, 75, 95)

sim = criar_sistema_triagem_uti()

risco_89 = avaliar_paciente(sim, 75, 89)

print(f"Risco para SpO2 de 95%: {risco_95:.2f}%")
print(f"Risco para SpO2 de 89%: {risco_89:.2f}%")
print(f"Diferenca entre os riscos: {risco_89 - risco_95:.2f} pontos percentuais")

Risco para SpO2 de 95%: 35.71%
Risco para SpO2 de 89%: 85.57%
Diferenca entre os riscos: 49.87 pontos percentuais


In [30]:

%%writefile test_uti_triagem.py

import pytest

from src.uti_triagem import (
    criar_sistema_triagem_uti,
    avaliar_paciente
)


@pytest.fixture
def simulador_uti():
    return criar_sistema_triagem_uti()


# MISSÃO 1: Paciente saudável com baixo risco
def test_paciente_saudavel_baixo_risco(simulador_uti):
    risco = avaliar_paciente(simulador_uti, 75, 98)

    print(f"\nMissão 1 - Risco do paciente saudável: {risco:.2f}%")

    assert risco < 30.0


# MISSÃO 2: Salto não linear da saturação
def test_salto_nao_linear_queda_spo2(simulador_uti):
    risco_95 = avaliar_paciente(simulador_uti, 75, 95)

    # Criar outro simulador para uma nova avaliação
    simulador = criar_sistema_triagem_uti()
    risco_89 = avaliar_paciente(simulador, 75, 89)

    print(f"\nMissão 2 - Risco com SpO2 de 95%: {risco_95:.2f}%")
    print(f"Missão 2 - Risco com SpO2 de 89%: {risco_89:.2f}%")
    print(f"Missão 2 - Aumento do risco: {risco_89 - risco_95:.2f} pontos percentuais")

    assert risco_95 < 35.0
    assert risco_89 > 65.0
    assert risco_89 > risco_95


# MISSÃO 3: Validação dos limites fisiológicos
def test_validacao_limites_fisiologicos(simulador_uti):
    with pytest.raises(ValueError):
        avaliar_paciente(simulador_uti, 39, 98)

    print("\nMissão 3 - FC abaixo de 40 bpm rejeitada corretamente.")

    simulador = criar_sistema_triagem_uti()

    with pytest.raises(ValueError):
        avaliar_paciente(simulador, 75, 101)

    print("Missão 3 - SpO2 acima de 100% rejeitada corretamente.")


Overwriting test_uti_triagem.py


In [31]:
!python -m pytest -v -s test_uti_triagem.py

============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: platformdirs-4.13.5, anyio-4.15.1, typeguard-4.6.0, langsmith-0.14.12
collected 3 items                                                              

test_uti_triagem.py::test_paciente_saudavel_baixo_risco 
Missão 1 - Risco do paciente saudável: 11.67%
PASSED
test_uti_triagem.py::test_salto_nao_linear_queda_spo2 
Missão 2 - Risco com SpO2 de 95%: 34.08%
Missão 2 - Risco com SpO2 de 89%: 85.57%
Missão 2 - Aumento do risco: 51.50 pontos percentuais
PASSED
test_uti_triagem.py::test_validacao_limites_fisiologicos 
Missão 3 - FC abaixo de 40 bpm rejeitada corretamente.
Missão 3 - SpO2 acima de 100% rejeitada corretamente.
PASSED

============================== 3 passed in 1.20s ===============================
